# Model inference on Kaggle / Colab

Choose a Hugging Face Transformers causal-language-model checkpoint or an Ollama model below. These are different runtimes: a Hub repository must support `AutoModelForCausalLM`, while Ollama requires its own server/model format. No single loader supports every architecture or model link. Check the model card, license, disk, and GPU memory requirements. Run cells in order. Server and tunnel cells are optional and must be started explicitly.

In [ ]:
%pip -q install -U transformers accelerate fastapi uvicorn requests pyngrok

In [ ]:
import os, platform, subprocess, time
from pathlib import Path
import torch

BACKEND = "huggingface"  # "huggingface" or "ollama"
HF_MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"  # Hub repo ID or https://huggingface.co/owner/repo
OLLAMA_MODEL = "qwen2.5:3b"
OLLAMA_BASE_URL = "http://127.0.0.1:11434"
from urllib.parse import urlparse
USE_HF_TOKEN = False  # set True for gated/private Hub models after adding HF_TOKEN to platform secrets
def platform_secret(name):
    if os.environ.get(name): return os.environ[name]
    try:
        from google.colab import userdata
    except ImportError: pass
    else: return userdata.get(name)
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError: pass
    else: return UserSecretsClient().get_secret(name)
    raise RuntimeError(f"Set {name} as an environment variable or notebook secret")
def normalize_hf_model_ref(reference):
    parsed = urlparse(reference)
    if not parsed.scheme: return reference
    if parsed.hostname not in {"huggingface.co", "www.huggingface.co"}:
        raise ValueError("Model URL must point to huggingface.co")
    parts = [part for part in parsed.path.split("/") if part]
    if len(parts) < 2: raise ValueError("Hub URL must identify an owner and model repository")
    return "/".join(parts[:2])
HF_MODEL_ID = normalize_hf_model_ref(HF_MODEL_ID)
HF_TOKEN = platform_secret("HF_TOKEN") if USE_HF_TOKEN else os.environ.get("HF_TOKEN")
print("Python:", platform.python_version(), "CUDA:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
if BACKEND not in {"huggingface", "ollama"}: raise ValueError("Unknown BACKEND")

## Ollama setup (only for `BACKEND = "ollama"`)

Linux Kaggle/Colab runtimes can start a local Ollama service. If you already have an Ollama endpoint, set `OLLAMA_BASE_URL` instead and skip this cell. Model downloads can exceed free runtime storage and memory.

In [ ]:
if BACKEND == "ollama":
    if platform.system() != "Linux": raise RuntimeError("Ollama setup cell expects Linux")
    if subprocess.run(["bash", "-lc", "command -v ollama"], capture_output=True).returncode:
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    ollama_log = open("ollama-serve.log", "w", encoding="utf-8")
    ollama_process = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT)
    import requests
    for _ in range(30):
        try:
            requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=2).raise_for_status()
            break
        except requests.RequestException: time.sleep(2)
    else: raise RuntimeError("Ollama did not start; inspect ollama-serve.log")
    subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True)

In [ ]:
import requests
model = tokenizer = None
if BACKEND == "huggingface":
    from transformers import AutoModelForCausalLM, AutoTokenizer
    dtype = (torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
             else torch.float16 if torch.cuda.is_available() else torch.float32)
    tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(
        HF_MODEL_ID, token=HF_TOKEN, torch_dtype=dtype, device_map="auto")
    if tokenizer.pad_token_id is None: tokenizer.pad_token = tokenizer.eos_token

def generate(prompt: str, max_new_tokens: int = 160) -> str:
    if not prompt.strip(): raise ValueError("prompt must not be empty")
    if BACKEND == "ollama":
        response = requests.post(f"{OLLAMA_BASE_URL}/api/generate",
            json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False,
                  "options": {"num_predict": max_new_tokens}}, timeout=600)
        response.raise_for_status()
        return response.json()["response"]
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.7, top_p=0.9,
            pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(generate("Explain transformer language models in three sentences."))

## Optional bearer-token-protected API

This cell starts an API only when run. Keep the generated bearer token private. Do not expose a public tunnel without authentication. Public tunnels are temporary, may be reachable by anyone with the URL, and may incur provider/runtime costs.

In [ ]:
import secrets, threading, uvicorn
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field
API_TOKEN = os.environ.get("NOTEBOOK_API_TOKEN") or secrets.token_urlsafe(32)
app = FastAPI(title="Notebook model API")
class GenerateRequest(BaseModel):
    prompt: str = Field(min_length=1, max_length=12000)
    max_new_tokens: int = Field(default=160, ge=1, le=1024)
@app.post("/generate")
def generate_endpoint(body: GenerateRequest, authorization: str | None = Header(default=None)):
    if authorization is None or not secrets.compare_digest(authorization, f"Bearer {API_TOKEN}"):
        raise HTTPException(status_code=401, detail="Unauthorized")
    return {"text": generate(body.prompt, body.max_new_tokens)}
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000))
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start(); time.sleep(2)
if not server_thread.is_alive(): raise RuntimeError("API server failed to start")
print("Keep this API bearer token private:", API_TOKEN)

## Optional ngrok tunnel

Add `NGROK_AUTH_TOKEN` through Kaggle/Colab secrets. Never commit or print the provider token. The generated public URL is temporary; keep it private and close the tunnel when done.

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token(platform_secret("NGROK_AUTH_TOKEN"))
ngrok_tunnel = ngrok.connect(8000, "http")
print("Temporary ngrok URL:", ngrok_tunnel.public_url)

## Optional Cloudflare Quick Tunnel

Install the `cloudflared` executable following Cloudflare's official instructions first. Quick Tunnels do not need an account and return a temporary public URL. Do not run both tunnel choices unless needed.

In [ ]:
import re, shutil
if shutil.which("cloudflared") is None:
    raise RuntimeError("Install cloudflared from Cloudflare's official documentation first")
cloudflared_log = open("cloudflared.log", "w", encoding="utf-8")
cloudflared_process = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=cloudflared_log, stderr=subprocess.STDOUT)
public_url = None
for _ in range(30):
    time.sleep(2); cloudflared_log.flush()
    log = Path("cloudflared.log").read_text(encoding="utf-8", errors="replace")
    match = re.search(r"https://[A-Za-z0-9.-]+\.trycloudflare\.com", log)
    if match: public_url = match.group(0); break
    if cloudflared_process.poll() is not None: raise RuntimeError("cloudflared exited; inspect log")
if public_url is None: raise TimeoutError("Tunnel URL not found; inspect cloudflared.log")
print("Temporary Cloudflare URL:", public_url)

Call `POST <tunnel-url>/generate` with JSON `{"prompt":"..."}` and header `Authorization: Bearer <API_TOKEN>`. After use, close the ngrok tunnel if started, terminate the Cloudflare process if started, and stop the API server (`server.should_exit = True`).

In [ ]:
# Local authenticated smoke test; replace the URL with your temporary tunnel in your own client.
response = requests.post("http://127.0.0.1:8000/generate",
    headers={"Authorization": f"Bearer {API_TOKEN}"},
    json={"prompt": "What is retrieval augmented generation?", "max_new_tokens": 80}, timeout=600)
response.raise_for_status(); print(response.json())
# Cleanup when finished:
# server.should_exit = True
# if "ngrok_tunnel" in globals(): ngrok.disconnect(ngrok_tunnel.public_url)
# if "cloudflared_process" in globals(): cloudflared_process.terminate()